In [0]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import *

schema = StructType([
    StructField("order_id", IntegerType()),
    StructField("product_name", StringType()),     
    StructField("product_category", StringType()),
    StructField("customer", StringType()),
    StructField("city", StringType()),
    StructField("amount", IntegerType()),
    StructField("timestamp", DoubleType())
])

df = (spark.readStream
      .format("kafka")
      .option("kafka.bootstrap.servers", "pkc-921jm.us-east-2.aws.confluent.cloud:9092")
      .option("subscribe", "stream_orders")
      .option("startingOffsets", "latest")
      .option("kafka.security.protocol", "SASL_SSL")
      .option("kafka.sasl.mechanism", "PLAIN")
      .option("kafka.sasl.jaas.config", "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username='NUZ6D4PCAKVJG6VM' password='cfltTfaOPTNMAk1I0FSTZrIlDAC3TBedQiPTrIA5TSelAjNb8yZgt7wnrLDmXsxg';")
      .load()
)

parsed_df = (df.selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)", "timestamp as kafka_timestamp")
              .select(col("key"), from_json(col("value"), schema).alias("data"), col("kafka_timestamp"))
              .select("key", "data.*", "kafka_timestamp")
)

query = (parsed_df.writeStream
         .format("delta")
         .outputMode("append")
         .trigger(availableNow=True)
         .option("checkpointLocation", "/Volumes/kafka_data/kafka_bronze/bronze_volume") # Ensure this path is set
         .table("kafka_data.kafka_bronze.kafka_raw_stream")
)

In [0]:
from pyspark.sql.functions import col, to_timestamp

bronze_df = spark.readStream.table("kafka_data.kafka_bronze.kafka_raw_stream")

silver_df = (
    bronze_df
    .filter(col("order_id").isNotNull())
    .filter(col("amount").isNotNull())
    .filter(col("amount") > 0)
    .withColumn("event_time", to_timestamp(col("timestamp")))
    .drop("timestamp") 
    .dropDuplicates(["order_id"])  
)

silver_query = (
    silver_df.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", "/Volumes/kafka_data/kafka_silver/silver_volume")
    .option("mergeSchema", "true")
    .table("kafka_data.kafka_silver.clean_orders")
)

In [0]:
from pyspark.sql.functions import sum, count, window

silver_stream = spark.readStream.table("kafka_data.kafka_silver.clean_orders")

gold_df = (
    silver_stream
    .withWatermark("event_time", "5 minutes")
    .groupBy(
        window(col("event_time"), "5 minutes"),
        col("city")
    )
    .agg(
        sum("amount").alias("total_sales"),
        count("order_id").alias("total_orders")
    )
)

gold_query = (
    gold_df.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", "/Volumes/kafka_data/kafka_gold/gold_volume")
    .table("kafka_data.kafka_gold.aggregated_sales")
)

spark.sql("USE kafka_data.kafka_bronze")

kafka_bootstrap = "pkc-921jm.us-east-2.aws.confluent.cloud:9092"
topic = "stream_orders"
api_key = "MUT2TJ3JCPJPWZUI"
api_secret = "cfltL36gOw6TDYwmd+O/NsNMGSBOTHogCsRxE0b8MmOMIYXzJhdOEJNvW7iTGYiA"

df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", kafka_bootstrap)
    .option("subscribe", topic)
    .option("startingOffsets", "latest")
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option(
      "kafka.sasl.jaas.config",
      "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username='MUT2TJ3JCPJPWZUI' password='cfltL36gOw6TDYwmd+O/NsNMGSBOTHogCsRxE0b8MmOMIYXzJhdOEJNvW7iTGYiA';"
  ) \
    .load()
)

from pyspark.sql.functions import col

kafka_df = df.selectExpr(
    "CAST(key AS STRING)",
    "CAST(value AS STRING)",
    "timestamp"
)

query = (
    kafka_df.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option(
        "checkpointLocation",
        "/Volumes/kafka_data/kafka_bronze/kafka_volume/kafka_check"
    )
    .table("kafka_bronze.kafka_raw_stream")
)

spark.streams.active

query.status


dbutils.fs.rm("/Volumes/kafka_data/kafka_bronze/kafka_volume/kafka_raw_stream", True)

spark.sql("DROP TABLE IF EXISTS kafka_data.kafka_bronze.kafka_raw_stream")